# 🚲 Pandas на реальних даних: EDA прокату велосипедів
## Практичний ноутбук із прикладами й домашніми завданнями

1. прочитайте пояснення 
2. **запустіть готовий приклад** 
3. **виконайте  завдання** 

## 0. Де взяти датасет і як підготувати файл

**Офіційне джерело:** [Bike Sharing, UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset).  
**Пряме посилання на ZIP:** [Bike Sharing Dataset.zip](https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip).

1. Самостійно завантажте архів за посиланням.
2. Розпакуйте його.
3. Знайдіть **`hour.csv`** — саме його використовуємо. Файл `day.csv` **не** потрібен.
4. Покладіть `hour.csv` **у ту саму папку, що й цей `.ipynb`**. У Google Colab — вручну завантажте `hour.csv` через ліву панель **Files**.
5. Виконайте наступну комірку. Якщо бачите `FileNotFoundError`, перевірте назву файлу та його розташування.

**Чому `hour.csv`?** Кожен рядок — одна година спостереження, а не одна конкретна оренда. Наприклад, `cnt = 250` означає *250 оренд протягом години*. Це важливо, бо **кількість рядків ≠ кількість оренд**.

**Джерело та ліцензія:** Fanaee-T, H. (2013), *Bike Sharing*, UCI, CC BY 4.0, [DOI](https://doi.org/10.24432/C5W894).

In [1]:
# Єдиний імпорт у цьому ноутбуці: тільки Pandas.
import pandas as pd

# Файл hour.csv ви завантажили самостійно й поклали поруч із ноутбуком.
# parse_dates перетворює текстову дату на зручний для аналізу тип datetime.
df = pd.read_csv("hour.csv", parse_dates=["dteday"])

print("Файл прочитано!")
print("Рядків і колонок:", df.shape)
df.head(3)

Файл прочитано!
Рядків і колонок: (17379, 17)


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32


## Словничок колонок: що саме ми аналізуємо

| Колонка | Значення | На що звернути увагу |
|---|---|---|
| `instant` | номер запису | це ID, а не кількість оренд |
| `dteday` | календарна дата | у нас є кілька годин на одну дату |
| `hr` | година доби: 0–23 | не плутайте з номером рядка |
| `yr` | `0` = 2011; `1` = 2012 | це категорія, а не «нульовий рік» |
| `mnth` | місяць: 1–12 | ціле число, але календарна категорія |
| `season` | 1 — зима, 2 — весна, 3 — літо, 4 — осінь | числові коди категорій |
| `weekday` | день тижня: 0–6 | 0 = неділя, 6 = субота |
| `holiday` | 1 — свято, 0 — ні | окрема ознака від `workingday` |
| `workingday` | 1 — робочий день, 0 — вихідний або свято | не рівнозначно `holiday` |
| `weathersit` | 1 — ясніше, 2 — хмарно/туманно, 3 — опади, 4 — складна погода | категорії, а не шкала з рівними інтервалами |
| `temp` | нормалізована температура повітря | для `hour.csv` у °C: `temp * 47 - 8` |
| `atemp` | нормалізована відчутна температура | для `hour.csv` у °C: `atemp * 66 - 16` |
| `hum` | відносна вологість, поділена на 100 | у %: `hum * 100` |
| `windspeed` | нормалізована швидкість вітру | не сприймайте як км/год без перетворення |
| `casual` | кількість оренд незареєстрованими клієнтами | число оренд за годину |
| `registered` | кількість оренд зареєстрованими клієнтами | число оренд за годину |
| `cnt` | усі оренди за годину | **`cnt = casual + registered`** |

**Важливі пастки EDA:**

- `cnt`, `casual`, `registered` пов'язані арифметично. Їхня кореляція не є несподіваним відкриттям.
- Середня кількість оренд **за годину** та **загальна кількість оренд за місяць** — різні метрики.
- Цифрові коди `season` і `weathersit` — категорії, тож кореляцію Пірсона з ними треба трактувати обережно.
- Наші висновки описують історичні спостереження 2011–2012 років, а не автоматично поточний ринок.

### Три правила виконання

1. Після кожного прикладу **не просто скопіюйте код**, а змініть колонки, умови або агрегацію за змістом завдання.
2. **Відповідь — це не тільки код.** Напишіть хоча б 1–3 речення з конкретним числом і правильними одиницями вимірювання.
3. Якщо бачите порожню клітинку для завдання з коментарями `TODO`, впишіть власний код **під коментарем**. Комірки з лише коментарями не падають під час запуску.

**Мініприклад правильного висновку (числа умовні):** «У групі A середня кількість оренд становить 150 на годину, а у групі B — 120. Різниця становить 30 оренд за годину; однак це порівняння не доводить, що сама ознака групи є причиною різниці».

---
# Частина 1. Знайомимося з таблицею

Почнемо з базових команд. Зазвичай це перший етап EDA: зрозуміти, які дані взагалі є, що означає один рядок, які типи значень і наскільки вони різноманітні.

## 1. Перегляд рядків: `head()`, `tail()`, `sample()`

**Навіщо це потрібно?** Перш ніж рахувати метрики, варто побачити живі приклади записів: чи схожі вони на те, що ви очікували?

`head(n)` показує перші `n` рядків. `tail(n)` — останні. `sample(n, random_state=42)` обирає випадкові рядки; фіксований `random_state` дозволяє повторити вибір. Перші рядки не завжди показують усе різноманіття датасету.

**Приклад — виконайте комірку та розгляньте результат:**

In [ ]:
# Приклад: подивимося на перші 4 погодинні записи.
df.head(4)

**✍️ Завдання 1 — тепер ви:**

1. Покажіть **останні 6 рядків**.  
2. Виведіть **5 випадкових рядків** так, щоб вибір був повторюваний.  
3. Порівняйте: чи всі приклади стосуються однієї дати?

> **Підказка:** `df.tail(...)` та `df.sample(...)`. Для показу двох таблиць у одній комірці можна написати `print(...)` або створити дві комірки.

In [3]:
df.tail(6)
df.sample(5, random_state=42)

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
12830,12831,2012-06-23,3,1,6,19,0,6,0,1,0.80,0.6970,0.27,0.1940,185,240,425
8688,8689,2012-01-02,1,1,1,20,1,1,0,1,0.24,0.2273,0.41,0.2239,5,83,88
7091,7092,2011-10-28,4,0,10,2,0,5,1,1,0.32,0.3030,0.66,0.2836,1,3,4
12230,12231,2012-05-29,2,1,5,19,0,2,1,1,0.78,0.7121,0.52,0.3582,69,457,526
431,432,2011-01-20,1,0,1,0,0,4,1,1,0.26,0.2273,0.56,0.3881,5,8,13


**Напишіть словами:** Чим випадкова вибірка корисна на відміну від перегляду лише початку файлу?

_Ваша відповідь: ..._ Допомагає краще зрозуміти структуру, є різноманітність

## 2. Розмір і назви колонок: `shape`, `columns`, `nunique()`

**Навіщо це потрібно?** Допомагає перевірити, що файл не порожній, знайти ключові поля та зрозуміти кількість унікальних значень.

`shape` — пара `(кількість рядків, кількість колонок)`. `columns` — назви колонок, `nunique()` — кількість різних значень. **Унікальні дати** відрізняються від кількості погодинних рядків.

**Приклад — виконайте комірку та розгляньте результат:**

In [ ]:
# Приклад: кількість колонок і унікальних дат.
print("Колонок:", df.shape[1])
print("Унікальних дат:", df["dteday"].nunique())

**✍️ Завдання 2 — тепер ви:**

1. Виведіть **усі назви колонок**.  
2. Скільки унікальних значень має `hr`? Чи це очікувано?  
3. Скільки унікальних значень має `instant`? Порівняйте з кількістю рядків.

> **Підказка:** Скористайтесь `df.columns`, `df["hr"].nunique()`, `df["instant"].nunique()`.

In [7]:
print(df.columns)
print(df["hr"].unique())
print(df["instant"].nunique())

Index(['instant', 'dteday', 'season', 'yr', 'mnth', 'hr', 'holiday', 'weekday',
       'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed',
       'casual', 'registered', 'cnt'],
      dtype='str')
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23]
17379


**Напишіть словами:** Чому кількість унікальних дат набагато менша від кількості рядків?

кожен рядок відповідає окремому запису за певну годину, а не за цілий день

## 3. Типи даних: `info()`, `dtypes`, `select_dtypes()`

**Навіщо це потрібно?** Перед аналізом потрібно знати, де числа, де дати, а де текст. Неправильний тип може зламати обчислення або спотворити сенс.

`info()` показує тип, кількість непорожніх значень та пам’ять. `dtypes` повертає типи колонок. Pandas може зберігати **категорії як цілі числа** — це не робить їх безперервними величинами.

**Приклад — виконайте комірку та розгляньте результат:**

In [ ]:
# Приклад: переглядаємо структуру таблиці.
df.info()

**✍️ Завдання 3 — тепер ви:**

1. Виведіть типи колонок `dteday`, `hr`, `cnt`, `temp`.  
2. Отримайте список **усіх числових колонок**.  
3. Назвіть **три колонки**, які технічно числові, але за змістом є категоріями.

> **Підказка:** Тип конкретної колонки — `df["cnt"].dtype`; вибір числових — `df.select_dtypes(include="number").columns`.

In [9]:
print( df["dteday"].dtype)
print( df["hr"].dtype)
print( df["cnt"].dtype)
print( df["temp"].dtype)
print(df.select_dtypes(include="number").columns.tolist())

datetime64[us]
int64
int64
float64
['instant', 'season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed', 'casual', 'registered', 'cnt']


**Напишіть словами:** Чому не можна трактувати середнє значення `season` як «середню пору року»?

_Ваша відповідь: ..._ бо це не медіана, що рахувала б вірніше. Середнє значення більш непередбачуване

## 4. Описові статистики: `describe()`, `mean()`, `median()`

**Навіщо це потрібно?** Середнє, медіана та крайні значення допомагають побачити типовий попит і нерівномірність розподілу.

`mean()` — арифметичне середнє, `median()` — число посередині відсортованого набору. `describe()` показує кількість, середнє, стандартне відхилення, мінімум, квартилі та максимум. Якщо середнє значно вище медіани, варто перевірити наявність великих значень.

**Приклад — виконайте комірку та розгляньте результат:**

In [10]:
# Приклад: статистики загального числа оренд за годину.
df["cnt"].describe()

count    17379.000000
mean       189.463088
std        181.387599
min          1.000000
25%         40.000000
50%        142.000000
75%        281.000000
max        977.000000
Name: cnt, dtype: float64

**✍️ Завдання 4 — тепер ви:**

1. Обчисліть **середню та медіанну** кількість оренд `casual` за годину.  
2. Знайдіть **мінімальне й максимальне** значення `registered`.  
3. Яка із двох груп — `casual` чи `registered` — має більший розкид значень? Підтвердьте числом.

> **Підказка:** `df["casual"].agg(["mean", "median", "std"])` поверне одразу декілька статистик.

In [12]:
print(df["casual"].agg(["mean", "median"]))
print(df["registered"].agg(["min", "max"]))
print(df[["casual", "registered"]].std())

mean      35.676218
median    17.000000
Name: casual, dtype: float64
min      0
max    886
Name: registered, dtype: int64
casual         49.305030
registered    151.357286
dtype: float64


**Напишіть словами:** Яка метрика краще описує «типову» годину для одного з типів клієнтів і чому?

_Ваша відповідь: ..._ Медіана, бо вона не залежить від різкої зміни числа.

## 5. Частоти категорій: `value_counts()`

**Навіщо це потрібно?** Так можна побачити, які категорії зустрічаються часто, а яких майже немає. Малочисельні групи варто аналізувати обережно.

`value_counts()` рахує частоти. `normalize=True` перетворює їх на частки від 0 до 1. Частка 0.25 означає 25% *рядків*, а не обов’язково 25% *оренд*.

**Приклад — виконайте комірку та розгляньте результат:**

In [13]:
# Приклад: скільки записів належить до кожного сезону.
print(df["season"].value_counts().sort_index())
print("Частки записів:")
print((df["season"].value_counts(normalize=True).sort_index() * 100).round(1))

season
1    4242
2    4409
3    4496
4    4232
Name: count, dtype: int64
Частки записів:
season
1    24.4
2    25.4
3    25.9
4    24.4
Name: proportion, dtype: float64


**✍️ Завдання 5 — тепер ви:**

1. Порахуйте, скільки годинних записів припадає на **кожну категорію погоди** (`weathersit`).  
2. Переведіть ці частки у відсотки.  
3. Якої категорії найменше? Чому її середній попит може бути ненадійним для висновків?

> **Підказка:** Спробуйте `df["weathersit"].value_counts(normalize=True) * 100`.

In [14]:
print(df["weathersit"].value_counts().sort_index())
print((df["weathersit"].value_counts(normalize=True).sort_index() * 100).round(1))
print(df["weathersit"].value_counts().idxmin())

weathersit
1    11413
2     4544
3     1419
4        3
Name: count, dtype: int64
weathersit
1    65.7
2    26.1
3     8.2
4     0.0
Name: proportion, dtype: float64
4


**Напишіть словами:** Скільки спостережень є в найрідкіснішій погодній категорії?

_Ваша відповідь: ..._ 4

---
# Частина 2. Відбір, сортування та якість

У реальних задачах аналітик постійно ставить уточнювальні питання до таблиці: «покажи лише вихідні», «знайди пікові години», «чи точно всі рядки коректні?».

## 6. Вибір частини даних: колонки та `loc`

**Навіщо це потрібно?** Часто для питання потрібні не всі 17 колонок, а лише кілька.

Список у подвійних квадратних дужках обирає **кілька колонок**. `loc[умова, список_колонок]` одночасно відбирає **рядки за умовою** та **потрібні колонки**. `iloc` обирає рядки/стовпці за позиціями, а не за назвами.

**Приклад — виконайте комірку та розгляньте результат:**

In [ ]:
# Приклад: дата, година і кількість оренд для перших п'яти рядків.
df.loc[:, ["dteday", "hr", "cnt"]].head(5)

**✍️ Завдання 6 — тепер ви:**

1. Покажіть **лише** `dteday`, `hr`, `temp`, `weathersit`, `cnt` для перших 10 рядків.  
2. Через `iloc` покажіть перші 3 рядки та перші 4 колонки.  
3. Поясніть різницю між вибором за назвою і за позицією.

> **Підказка:** `df.loc[:, [...]]` для назв, `df.iloc[:3, :4]` для позицій.

In [16]:
print(df.loc[:, ["dteday", "hr", "temp", "weathersit", "cnt"]].head(10))
print(df.iloc[:3, :4])
#лок для рядків і колонок за назвами
#ілок для вибору з чисел

      dteday  hr  temp  weathersit  cnt
0 2011-01-01   0  0.24           1   16
1 2011-01-01   1  0.22           1   40
2 2011-01-01   2  0.22           1   32
3 2011-01-01   3  0.24           1   13
4 2011-01-01   4  0.24           1    1
5 2011-01-01   5  0.24           2    1
6 2011-01-01   6  0.22           1    2
7 2011-01-01   7  0.20           1    3
8 2011-01-01   8  0.24           1    8
9 2011-01-01   9  0.32           1   14
   instant     dteday  season  yr
0        1 2011-01-01       1   0
1        2 2011-01-01       1   0
2        3 2011-01-01       1   0


**Напишіть словами:** Коли відбір колонок робить результат зручнішим для читання?

_Ваша відповідь: ..._ Коли треба вивести лише певну кількість колонок для завдання

## 7. Фільтрація: просте порівняння

**Навіщо це потрібно?** Майже всі бізнес-запитання стосуються підмножин даних: робочі дні, певний рік, дощові години.

Умова `df["workingday"] == 1` утворює маску з `True/False` для кожного рядка. Якщо передати її в `df[...]`, залишаться лише рядки з `True`. Після фільтра можна обчислювати метрики.

**Приклад — виконайте комірку та розгляньте результат:**

In [ ]:
# Приклад: лише робочі дні.
work = df[df["workingday"] == 1]
print("Погодинних записів у робочі дні:", len(work))
print("Середні оренди за годину:", round(work["cnt"].mean(), 1))

**✍️ Завдання 7 — тепер ви:**

1. Відберіть **лише святкові дні** (`holiday == 1`).  
2. Порахуйте, скільки погодинних записів відповідають цій умові.  
3. Знайдіть медіану `cnt` для святкових днів.

> **Підказка:** Створіть нову змінну на кшталт `holidays = df[df["holiday"] == 1]`.

In [20]:
holidays = df[df["holiday"] == 1]
print(len(holidays))
print(holidays["cnt"].median())

500
97.0


**Напишіть словами:** Чому кількість рядків зі святами не дорівнює кількості святкових дат?

_Ваша відповідь: ..._ одна святкова дата може містити до 24 погодинних записів.

## 8. Складні умови: `&`, `|`, `between()`, `isin()`

**Навіщо це потрібно?** Окрема умова рідко відповідає на справжнє питання. Частіше цікавить поєднання години, погоди й типу дня.

`&` означає **і**, `|` — **або**. Кожну умову ставте в круглі дужки. `between(7, 9)` включає межі 7 та 9. `isin([...])` перевіряє входження у список.

**Приклад — виконайте комірку та розгляньте результат:**

In [ ]:
# Приклад: ранкові години 7–9 у робочі дні.
morning_work = df[(df["hr"].between(7, 9)) & (df["workingday"] == 1)]
print("Рядків:", len(morning_work))
print("Середній cnt:", morning_work["cnt"].mean())

**✍️ Завдання 8 — тепер ви:**

1. Виберіть **неробочі дні**, години **17–20 включно**.  
2. Порахуйте середнє `cnt` у цій групі.  
3. Окремо виберіть усі записи, де погода має код **2 або 3**. Скільки їх?

> **Підказка:** Спробуйте `df[(...) & (...)]` та `df["weathersit"].isin([2, 3])`.

In [22]:
evening_nonwork = df[(df["workingday"] == 0) & (df["hr"].between(17, 20))]
print(len(evening_nonwork))
print(round(evening_nonwork["cnt"].mean(), 1))
bad_weather = df[df["weathersit"].isin([2, 3])]

print(len(bad_weather))

921
252.8
5963


**Напишіть словами:** Як змінюється сенс порівняння, якщо одночасно враховувати і тип дня, і час?

_Ваша відповідь: ..._Точнішим, бо попит більше або менше, залежно від того чи це робочий чи вихідний

## 9. Сортування і пошук крайніх значень

**Навіщо це потрібно?** Так можна швидко знайти години рекордного попиту, найбільш холодні спостереження чи підозрілі значення.

`sort_values("назва")` впорядковує рядки. `ascending=False` — від найбільшого до найменшого. `nlargest(n, "колонка")` швидко повертає `n` найбільших значень. Важливо дивитися **інші колонки цих рядків**, щоб зрозуміти контекст.

**Приклад — виконайте комірку та розгляньте результат:**

In [ ]:
# Приклад: 5 найтепліших спостережень (за нормалізованою temp).
df.nlargest(5, "temp")[["dteday", "hr", "temp", "cnt"]]

**✍️ Завдання 9 — тепер ви:**

1. Знайдіть **10 годин із найбільшою кількістю оренд**.  
2. Виведіть для них дату, годину, погоду та `cnt`.  
3. Знайдіть **5 годин із найменшою кількістю оренд**. Як ви пояснили б ці значення?

> **Підказка:** `df.nlargest(10, "cnt")[[...]]` і `df.nsmallest(5, "cnt")`.

In [38]:
top_10 = df.nlargest(10, "cnt")

print(top_10[["dteday", "hr", "weathersit", "cnt"]])
bottom_5 = df.nsmallest(5, "cnt")

bottom_5[["dteday", "hr", "weathersit", "cnt"]]

          dteday  hr  weathersit  cnt
14773 2012-09-12  18           1  977
14964 2012-09-20  17           1  976
14748 2012-09-11  17           1  970
14725 2012-09-10  18           1  968
15084 2012-09-25  17           1  967
15780 2012-10-24  17           1  963
10622 2012-03-23  17           2  957
15108 2012-09-26  17           1  953
15444 2012-10-10  17           1  948
15588 2012-10-16  17           1  943


,dteday,hr,weathersit,cnt
4,2011-01-01,4,1,1
5,2011-01-01,5,2,1
30,2011-01-02,7,2,1
49,2011-01-03,4,1,1
71,2011-01-04,2,1,1


**Напишіть словами:** Чому рекордні години не завжди означають помилки у даних?

_Ваша відповідь: ..._вони можуть відображати реальні періоди підвищеного попиту.

## 10. Пропуски, дублікати та перевірка логіки

**Навіщо це потрібно?** EDA — це не лише пошук закономірностей. Спочатку потрібно переконатись, що базові співвідношення в таблиці не порушені.

`isna().sum()` рахує порожні значення за колонками. `duplicated().sum()` рахує повторення **повних рядків**, але два рядки за ту саму дату — не обов’язково дублікати, адже години різні. Якість даних включає також перевірки бізнес-правил.

**Приклад — виконайте комірку та розгляньте результат:**

In [ ]:
# Приклад: кількість пропущених значень за колонками.
missing = df.isna().sum().sort_values(ascending=False)
print(missing.head(10))

**✍️ Завдання 10 — тепер ви:**

1. Порахуйте **повні дублікати**.  
2. Перевірте, чи є повторні поєднання **дата + година**.  
3. Порахуйте рядки, де **`cnt != casual + registered`**.  
4. Знайдіть, чи є `cnt` меншим за нуль.

> **Підказка:** Для правила суми: `(df["cnt"] != df["casual"] + df["registered"]).sum()`.

In [24]:
print(df.duplicated().sum())
print(df.duplicated(subset=["dteday", "hr"]).sum())
print((df["cnt"] != df["casual"] + df["registered"]).sum())
print((df["cnt"] < 0).sum())

0
0
0
0


**Напишіть словами:** Навіть якщо проблем немає, чому важливо було їх перевірити?

_Ваша відповідь: ..._це допомагає переконатися в їхній правильності

## 11. Нові колонки: перетворюємо шкали

**Навіщо це потрібно?** Нормалізовані числа 0–1 складніше інтерпретувати. У EDA часто варто додати зрозуміліші одиниці вимірювання.

Створення `df["нова"] = ...` додає колонку. У вихідному `hour.csv` `temp` нормалізовано за шкалою від −8 до +39 °C, а `hum` — поділено на 100. Виконуйте перетворення в **нових колонках**, а не перезаписуйте оригінальні.

**Приклад — виконайте комірку та розгляньте результат:**

In [ ]:
# Приклад: температура у градусах Цельсія.
df["temp_c"] = df["temp"] * 47 - 8
df[["temp", "temp_c"]].head()

**✍️ Завдання 11 — тепер ви:**

1. Створіть колонку `humidity_pct` — вологість у **відсотках**.  
2. Створіть `feels_like_c` — відчутну температуру у **°C**.  
3. Покажіть 5 рядків із `hum`, `humidity_pct`, `atemp`, `feels_like_c`.

> **Підказка:** Формули є у словнику колонок; значення для відчутної температури: діапазон від −16 до +50 °C.

In [26]:
df["humidity_pct"] = df["hum"] * 100
df["feels_like_c"] = df["atemp"] * 66 - 16
df[["hum", "humidity_pct", "atemp", "feels_like_c"]].head(5)

,hum,humidity_pct,atemp,feels_like_c
0,0.81,81.0,0.2879,3.0014
1,0.80,80.0,0.2727,1.9982
2,0.80,80.0,0.2727,1.9982
3,0.75,75.0,0.2879,3.0014
4,0.75,75.0,0.2879,3.0014


**Напишіть словами:** Навіщо в бізнес-звіті показувати 20 °C замість нормалізованого значення?

_Ваша відповідь: ..._така величина зрозуміла та звична для більшості людей.

---
# Частина 3. Шукаємо закономірності

Тепер ми вже вміємо брати потрібні записи й перевіряти якість. Далі — найкорисніша частина Pandas для аналітики: порівнювати групи та перетин двох ознак.

## 12. Просте групування: `groupby()`

**Навіщо це потрібно?** Питання «в яку пору року попит вищий?» не вирішити простим `mean()` по всій таблиці — треба рахувати середнє **окремо для груп**.

`groupby("season")["cnt"].mean()` формує одну середню величину **на сезон**. `mean()` — середня кількість оренд *за спостережену годину*, а `sum()` — загальна кількість оренд у всіх спостережених годинах. Вибір метрики принциповий.

**Приклад — виконайте комірку та розгляньте результат:**

In [ ]:
# Приклад: середні оренди за годину в кожному сезоні.
seasonal_mean = df.groupby("season")["cnt"].mean().round(1)
seasonal_mean

**✍️ Завдання 12 — тепер ви:**

1. Розрахуйте середню кількість оренд **за кожної погодної категорії**.  
2. Відсортуйте результат від більшого до меншого.  
3. Зробіть аналогічне порівняння **за роками** (`yr`).

> **Підказка:** Після групування можна додати `.sort_values(ascending=False).round(1)`.

In [37]:
weather_mean = df.groupby("weathersit")["cnt"].mean().sort_values(ascending=False).round(1)

year_mean = df.groupby("yr")["cnt"].mean().sort_values(ascending=False).round(1)

year_mean

yr
1    234.7
0    143.8
Name: cnt, dtype: float64

**Напишіть словами:** Де середній попит вищий? Чому таке порівняння ще не доводить причинного впливу погоди?

_Ваша відповідь: ..._оскільки на кількість оренд могли впливати й інші фактори

## 13. Кілька статистик одразу: `agg()`

**Навіщо це потрібно?** Одна середня величина приховує кількість спостережень і нерівномірність. Корисно дивитись на середнє, медіану, мінімум і кількість.

`agg()` дозволяє називати вихідні показники. `size` рахує **рядки**, `mean` — середнє, `median` — медіану, `sum` — суму. Не плутайте `size` і кількість оренд.

**Приклад — виконайте комірку та розгляньте результат:**

In [ ]:
# Приклад: порівнюємо робочі й неробочі дні.
df.groupby("workingday").agg(
    hours=("cnt", "size"),
    avg_per_hour=("cnt", "mean"),
    median_per_hour=("cnt", "median"),
    total_rentals=("cnt", "sum"),
).round(1)

**✍️ Завдання 13 — тепер ви:**

1. Побудуйте таблицю **за годинами `hr`**, де є: кількість спостережень, середній `cnt`, медіанний `cnt`, максимальний `cnt`.  
2. Визначте три години з найвищим **середнім** попитом.  
3. Поясніть, чому рейтинг за середнім може відрізнятися від рейтингу за сумою.

> **Підказка:** Візьміть назви вихідних показників за зразком та додайте `max_per_hour=("cnt", "max")`.

In [36]:
hourly_stats = df.groupby("hr").agg(
    observations=("cnt", "size"),
    avg_cnt=("cnt", "mean"),
    median_cnt=("cnt", "median"),
    max_cnt=("cnt", "max")
).round(1)
hourly_stats.sort_values(
    by="avg_cnt",
    ascending=False
).head(3)

,observations,avg_cnt,median_cnt,max_cnt
hr,,,,
17,730,461.5,475.0,976
18,728,425.5,418.5,977
8,727,359.0,385.0,839


**Напишіть словами:** Які години найважливіші для планування кількості доступних велосипедів?

_Ваша відповідь: ..._ 17, 18, 8

## 14. Дві ознаки одночасно: `groupby([...])`

**Навіщо це потрібно?** Групування тільки за одним фактором може маскувати різницю між групами. Наприклад, одна й та сама година поводиться по-різному у робочий та неробочий день.

Якщо передати до `groupby` список колонок, Pandas створить групи для **кожної комбінації**. `reset_index()` повертає колонки групування зі спеціального індексу в звичайні колонки.

**Приклад — виконайте комірку та розгляньте результат:**

In [ ]:
# Приклад: середні оренди за годинами і типом дня.
by_hour_day = (
    df.groupby(["workingday", "hr"])["cnt"]
      .mean()
      .round(1)
      .reset_index(name="avg_rentals")
)
by_hour_day.head(8)

**✍️ Завдання 14 — тепер ви:**

1. Порахуйте середнє `cnt` для **кожної пари сезон + погода**.  
2. Додайте кількість рядків у групі (через `agg`).  
3. Знайдіть комбінації з високим середнім попитом і перевірте, чи достатньо там спостережень.

> **Підказка:** `groupby(["season", "weathersit"]).agg(avg=("cnt", "mean"), n=("cnt", "size"))`.

In [34]:
season_weather = df.groupby(["season", "weathersit"]).agg(
    avg=("cnt", "mean"),
    n=("cnt", "size")
).round(1)

print(season_weather)

season_weather.sort_values(by="avg", ascending=False)

                     avg     n
season weathersit             
1      1           122.1  2665
       2           103.1  1205
       3            58.7   369
       4            74.3     3
2      1           232.4  2859
       2           181.3  1144
       3           115.2   406
3      1           244.5  3280
       2           224.8   947
       3           172.0   269
4      1           209.5  2609
       2           201.5  1248
       3           116.5   375


avg     n
season weathersit             
3      1           244.5  3280
2      1           232.4  2859
3      2           224.8   947
4      1           209.5  2609
       2           201.5  1248
2      2           181.3  1144
3      3           172.0   269
1      1           122.1  2665
4      3           116.5   375
2      3           115.2   406
1      2           103.1  1205
       4            74.3     3
       3            58.7   369

**Напишіть словами:** Чому середнє, обчислене за 2 годинами, менш стабільне, ніж за 200 годинами?

_Ваша відповідь: ..._Чим менше годин, тим нестабільніше, бо залежить від різних зростань чи спадань

## 15. Таблиця «рядки × стовпці»: `pivot_table()`

**Навіщо це потрібно?** Зі списку груп не завжди легко порівнювати значення. Pivot-таблиця створює компактну матрицю: по рядках один фактор, по колонках — інший.

`pivot_table(index=..., columns=..., values=..., aggfunc="mean")` показує середнє для кожного перетину. Це не новий спосіб обчислення середнього, а **зручна форма** групування.

**Приклад — виконайте комірку та розгляньте результат:**

In [ ]:
# Приклад: години × тип дня, середня кількість оренд.
df.pivot_table(
    index="hr", columns="workingday", values="cnt", aggfunc="mean"
).round(1)

**✍️ Завдання 15 — тепер ви:**

1. Створіть таблицю **місяць × рік**, у якій записане середнє `cnt` за годину.  
2. Для яких місяців різниця між 2011 та 2012 роками найбільша?  
3. В окремій таблиці порівняйте **сезон × тип дня**.

> **Підказка:** Після побудови `table` відніміть `table[0]` від `table[1]`, якщо обидві колонки присутні.

In [41]:
table = df.pivot_table(
    index="mnth",
    columns="yr",
    values="cnt",
    aggfunc="mean"
).round(1)

print(table)

difference = (table[1] - table[0]).abs().sort_values(ascending=False)

print(difference)

print(difference.idxmax())
print(difference.max())

season_day = df.pivot_table(
    index="season",
    columns="workingday",
    values="cnt",
    aggfunc="mean"
).round(1)

season_day

yr        0      1
mnth              
1      55.5  130.6
2      74.3  149.0
3      87.7  221.9
4     131.9  242.7
5     182.6  263.3
6     199.3  281.7
7     190.0  273.7
8     187.0  288.3
9     177.7  303.6
10    166.2  280.8
11    142.1  212.6
12    117.8  166.7
mnth
3     134.2
9     125.9
10    114.6
4     110.8
8     101.3
7      83.7
6      82.4
5      80.7
1      75.1
2      74.7
11     70.5
12     48.9
dtype: float64
3
134.2


workingday,0,1
season,,
1,95.1,119.4
2,214.4,205.7
3,230.3,238.5
4,191.9,202.2


**Напишіть словами:** Чому порівнювати ті самі місяці двох років справедливіше, ніж довільні місяці?

_Ваша відповідь: ..._попит на велосипеди залежить від сезону та погодних умов

## 16. Частоти перетинів: `crosstab()`

**Навіщо це потрібно?** Іноді нас цікавлять не середні оренди, а **розподіл спостережень**: скільки годин було ясних чи дощових у різні сезони.

`pd.crosstab()` рахує комбінації категорій. `normalize="index"` ділить кожну клітинку на суму її **рядка**. Тому відсотки в кожному рядку дорівнюють 100%. Це інша постановка питання, ніж середній `cnt`.

**Приклад — виконайте комірку та розгляньте результат:**

In [ ]:
# Приклад: частка погодних категорій у кожному сезоні.
pd.crosstab(
    df["season"], df["weathersit"], normalize="index"
).mul(100).round(1)

**✍️ Завдання 16 — тепер ви:**

1. Створіть **таблицю кількості** спостережень для `workingday × weathersit`.  
2. Створіть **таблицю відсотків у межах кожного типу дня**.  
3. Поясніть, що означає одна обрана вами клітинка (який знаменник?).

> **Підказка:** `pd.crosstab(df["workingday"], df["weathersit"], normalize="index")`.

In [43]:
weather_counts = pd.crosstab(
    df["workingday"],
    df["weathersit"]
)

weather_percent = pd.crosstab(
    df["workingday"],
    df["weathersit"],
    normalize="index"
).mul(100).round(1)

weather_percent

weathersit,1,2,3,4
workingday,,,,
0,68.6,24.5,6.9,0.0
1,64.3,26.9,8.8,0.0


**Напишіть словами:** Чим «30% годин були дощовими» відрізняється від «за дощу було 30% усіх оренд»?

_Ваша відповідь: ...«30% годин були дощовими» - 30% усіх годинних спостережень припадали на дощову погоду, другий - оренда була 30% при дощі

## 17. Два типи клієнтів: працюємо з кількома колонками

**Навіщо це потрібно?** Компанії потрібно розуміти не лише весь попит, а й того, хто цими велосипедами користується.

`sum()` за колонкою рахує всі оренди певного типу. Коли сума береться для кількох колонок, результат — окреме число для кожної з них. Під час порівняння годин краще застосовувати середнє **за годину**, щоб різна кількість записів не вводила в оману.

**Приклад — виконайте комірку та розгляньте результат:**

In [53]:
# Приклад: загальні кількості оренд за весь період.
segment_totals = df[["casual", "registered"]].sum()
print(segment_totals)
print("Частка casual, %:", round(segment_totals["casual"] / segment_totals.sum() * 100, 1))

casual         620017
registered    2672662
dtype: int64
Частка casual, %: 18.8


**✍️ Завдання 17 — тепер ви:**

1. Порахуйте середні `casual` і `registered` **за кожну годину доби** (дві колонки в таблиці).  
2. Знайдіть, о котрій годині в кожного сегмента найбільший середній попит.  
3. Повторіть аналіз, розділивши на робочі та неробочі дні.

> **Підказка:** Після `.groupby("hr")[["casual", "registered"]].mean()` можна викликати `.idxmax()`.

In [54]:
hourly_segments = df.groupby("hr")[["casual", "registered"]].mean().round(1)

print(hourly_segments["casual"].idxmax())
print(hourly_segments["registered"].idxmax())
day_segments = df.groupby(["workingday", "hr"])[["casual", "registered"]].mean().round(1)


peak_hours = day_segments.groupby(level="workingday").idxmax()

peak_hours

14
17


,casual,registered
workingday,,
0,"(0, 14)","(0, 12)"
1,"(1, 17)","(1, 17)"


**Напишіть словами:** Чим можуть пояснюватися відмінності між зареєстрованими та незареєстрованими? Це гіпотеза, а не доведена причина.

_Ваша відповідь: ..._можуть пояснюватися різними цілями поїздок. Зареєстровані наприклад для роботи, інші для відпочинку

---
# Частина 4. Складніша аналітика без графіків

Ці прийоми допомагають перейти від очевидних підрахунків до перевірки гіпотез: побачити зв’язок, незвичні значення й поведінку в часі.

## 18. Кореляції: `corr()` та їх обмеження

**Навіщо це потрібно?** Кореляція дозволяє кількісно оцінити лінійний зв’язок між двома числовими величинами.

Кореляція Пірсона близька до +1 за сильного позитивного лінійного зв’язку, близька до −1 за негативного і близька до 0 за слабкого **лінійного** зв’язку. Це **не** доказ причинності. Також не варто бездумно включати ID, кодовані категорії та `casual`/`registered` при аналізі `cnt`.

**Приклад — виконайте комірку та розгляньте результат:**

In [60]:
# Приклад: як кілька погодних показників пов'язані з cnt.
(df[["temp_c", "hum", "windspeed", "cnt"]]
    .corr()["cnt"]
    .sort_values(ascending=False)
    .round(3))


cnt          1.000
temp_c       0.405
windspeed    0.093
hum         -0.323
Name: cnt, dtype: float64

**✍️ Завдання 18 — тепер ви:**

1. Порахуйте кореляцію `cnt` окремо з **`temp_c` і `humidity_pct`**.  
2. Порахуйте кореляцію між `temp_c` і `feels_like_c`.  
3. Поясніть, чому `cnt` та `registered` не варто називати незалежним відкриттям.

> **Підказка:** Використовуйте створені в завданні 11 колонки та `df[[...]].corr()`.

In [63]:
weather_corr = df[["temp_c", "humidity_pct", "cnt"]].corr()


print(weather_corr.loc["cnt", "temp_c"].round(3))


print(weather_corr.loc["cnt", "humidity_pct"].round(3))


temp_corr = df[["temp_c", "feels_like_c"]].corr()
print(temp_corr.loc["temp_c", "feels_like_c"].round(3))

0.405
-0.323
0.988


**Напишіть словами:** Які є альтернативні пояснення кореляції між температурою й попитом (наприклад, сезон)?

_Ваша відповідь: ..._в теплі місяці люди частіше користуються велосипедами через кращу погоду

## 19. Квартилі, перцентилі та пошук викидів

**Навіщо це потрібно?** Знайти рекордну годину легко; складніше визначити, що означає «незвично висока». Перцентилі дозволяють зробити це прозоро.

`quantile(0.25)` — перший квартиль Q1; `quantile(0.75)` — третій квартиль Q3; `IQR = Q3 − Q1`. Поширене правило для пошуку кандидатів у викиди: значення нижче `Q1−1.5×IQR` або вище `Q3+1.5×IQR`. Це **сигнал для перевірки**, не наказ видалити рядок.

**Приклад — виконайте комірку та розгляньте результат:**

In [ ]:
# Приклад: дивимося на квартилі та 95-й перцентиль.
df["cnt"].quantile([0.25, 0.5, 0.75, 0.95])

**✍️ Завдання 19 — тепер ви:**

1. Обчисліть Q1, Q3 та IQR для `cnt`.  
2. Порахуйте нижню й верхню межі за правилом 1.5×IQR.  
3. Скільки погодинних записів лежить **за межами**?  
4. Виведіть 5 найвищих таких записів разом із датою та годиною.

> **Підказка:** Для Q1: `df["cnt"].quantile(0.25)`; умови з `|` ставте в окремі дужки.

In [70]:
q1 = df["cnt"].quantile(0.25)
q3 = df["cnt"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
outliers = df[(df["cnt"] < lower) |(df["cnt"] > upper)]
print(len(outliers))
outliers.nlargest(5, "cnt")[["dteday", "hr", "cnt"]]

505


,dteday,hr,cnt
14773,2012-09-12,18,977
14964,2012-09-20,17,976
14748,2012-09-11,17,970
14725,2012-09-10,18,968
15084,2012-09-25,17,967


**Напишіть словами:** Чому пікова година в будній день може бути статистичним викидом, але не помилкою даних?

_Ваша відповідь: ..._кількість оренд у цей час значно перевищує типові значення

## 20. Дати та щоденна агрегація

**Навіщо це потрібно?** Погодинні записи зручні для аналізу години пік, але питання «який день був найуспішнішим?» потребує зведення до рівня дня.

`groupby("dteday")["cnt"].sum()` обчислює всі оренди **за спостережені години одного дня**. Паралельно корисно рахувати, скільки годин є в кожному дні: відсутні години роблять порівняння денних сум менш чесним.

**Приклад — виконайте комірку та розгляньте результат:**

In [72]:
# Приклад: кількість оренд та спостережених годин за день.
daily = (
    df.groupby("dteday")
      .agg(total_rentals=("cnt", "sum"), observed_hours=("hr", "size"))
      .sort_index()
)
daily.head()

,total_rentals,observed_hours
dteday,,
2011-01-01,985,24
2011-01-02,801,23
2011-01-03,1349,22
2011-01-04,1562,23
2011-01-05,1600,23


**✍️ Завдання 20 — тепер ви:**

1. Визначте **5 днів із найбільшою сумою оренд**.  
2. Скільки днів мають **менше 24 погодинних записів**?  
3. Обчисліть для кожного дня **середню кількість оренд за спостережену годину**.  
4. Чи однакові лідери за денною сумою і за середнім на годину?

> **Підказка:** Використовуйте таблицю `daily`, створену в прикладі; нову метрику додайте як колонку.

In [74]:
daily.nlargest(5, "total_rentals")
days_less_24 = (daily["observed_hours"] < 24).sum()

print("Кількість днів із менше ніж 24 записами:", days_less_24)
daily["avg_rentals_per_hour"] = (
    daily["total_rentals"] / daily["observed_hours"]
)

print(daily.nlargest(1, "total_rentals"))
print(daily.nlargest(1, "avg_rentals_per_hour"))

Кількість днів із менше ніж 24 записами: 76
            total_rentals  observed_hours  avg_rentals_per_hour
dteday                                                         
2012-09-15           8714              24            363.083333
            total_rentals  observed_hours  avg_rentals_per_hour
dteday                                                         
2012-09-15           8714              24            363.083333


**Напишіть словами:** Чому день із 20 спостереженими годинами може виглядати гірше за день із 24 годинами?

_Ваша відповідь: ..._ тому що години, яких не вистачило, не рахуються.

## 21. Ковзне середнє: `rolling()`

**Навіщо це потрібно?** Щоденний попит шумний: на нього впливають конкретна погода, день тижня та випадкові події. Ковзне середнє згладжує коливання.

`rolling(7).mean()` обчислює середнє за поточним і попередніми 6 **рядками**. Після добової агрегації це зазвичай 7 дат, але треба перевірити, чи дат немає пропущених. `min_periods=7` не дозволяє рахувати неповне 7-денне вікно.

**Приклад — виконайте комірку та розгляньте результат:**

In [75]:
# Приклад: щоденна сума і ковзне середнє за 7 наявних дат.
daily = daily.sort_index().copy()
daily["rolling_7"] = daily["total_rentals"].rolling(7, min_periods=7).mean()
daily[["total_rentals", "rolling_7"]].head(12).round(1)

,total_rentals,rolling_7
dteday,,
2011-01-01,985,NaN
2011-01-02,801,NaN
2011-01-03,1349,NaN
2011-01-04,1562,NaN
2011-01-05,1600,NaN
2011-01-06,1606,NaN
2011-01-07,1510,1344.7
2011-01-08,959,1341.0
2011-01-09,822,1344.0


**✍️ Завдання 21 — тепер ви:**

1. Створіть **14-денне ковзне середнє** для `total_rentals`.  
2. Знайдіть дату максимального значення цього середнього.  
3. Порівняйте його зі звичайним максимумом денних оренд.  
4. Поясніть, чому перші 13 значень 14-денного вікна можуть бути порожніми.

> **Підказка:** Спробуйте `daily["total_rentals"].rolling(14, min_periods=14).mean()`.

In [78]:
daily = daily.sort_index().copy()

daily["rolling_14"] = (
    daily["total_rentals"].rolling(14, min_periods=14).mean()
)

daily[["total_rentals", "rolling_14"]].head(20).round(1)
max_rolling_date = daily["rolling_14"].idxmax()
max_rolling_value = daily["rolling_14"].max()

print(max_rolling_date)
print( round(max_rolling_value, 1))
max_daily_date = daily["total_rentals"].idxmax()
max_daily_value = daily["total_rentals"].max()

print( max_daily_date)
print(max_daily_value)

2012-09-22 00:00:00
7576.0
2012-09-15 00:00:00
8714


**Напишіть словами:** Чим короткий сплеск відрізняється від тривалого підвищення попиту?

_Ваша відповідь: ..._ сплеск це різке збільшення попиту, за коротку кількість.

---
# Частина 5. Додаткові виклики (за бажанням)

Ці задачі ближчі до реальної аналітичної роботи: порівнюємо спостереження з типовим значенням **для схожих умов**, а не зі всім датасетом одразу.

## 22. `transform()`: порівняння кожної години з типовою

**Навіщо це потрібно?** Годину о 8:00 недоцільно порівнювати із середнім за всі 24 години: будні й ночі мають різний режим попиту.

На відміну від `groupby().mean()`, який повертає одну відповідь на групу, `transform("mean")` повертає **значення для кожного початкового рядка**: середнє його групи. Так можна створити базову лінію та відхилення.

**Приклад — виконайте комірку та розгляньте результат:**

In [ ]:
# Приклад: порівнюємо кожний рядок із середнім для цієї ж години доби.
df["typical_for_hour"] = df.groupby("hr")["cnt"].transform("mean")
df["delta_from_hour"] = df["cnt"] - df["typical_for_hour"]
df[["dteday", "hr", "cnt", "typical_for_hour", "delta_from_hour"]].head()

**✍️ Завдання 22 — тепер ви:**

1. Створіть базову метрику для **комбінації `workingday + hr`**.  
2. Обчисліть відхилення кожного запису від середнього своєї групи.  
3. Знайдіть 5 найбільш **негативних відхилень**.  
4. Припустіть, чому вони могли виникнути (це лише гіпотези).

> **Підказка:** Перенесіть приклад, замінивши одну групувальну колонку на список із двох.

In [80]:
df["typical_for_group"] = (
    df.groupby(["workingday", "hr"])["cnt"].transform("mean")
)
df["delta_from_group"] = (
    df["cnt"] - df["typical_for_group"]
)
df.nsmallest(5, "delta_from_group")[
    ["dteday", "workingday", "hr", "cnt",
     "typical_for_group", "delta_from_group"]
]

,dteday,workingday,hr,cnt,typical_for_group,delta_from_group
586,2011-01-26,1,17,26,525.290581,-499.290581
17205,2012-12-24,1,17,26,525.290581,-499.290581
8065,2011-12-07,1,17,52,525.290581,-473.290581
17252,2012-12-26,1,17,53,525.290581,-472.290581
17206,2012-12-24,1,18,26,492.226908,-466.226908


**Напишіть словами:** Чому порівняння з нормою «та сама година + той самий тип дня» точніше?

_Ваша відповідь: ..._оскільки попит на велосипеди відрізняється залежно від часу доби та того, чи є день робочим

## 23. Перевірка гіпотези з `assign()` та `query()`

**Навіщо це потрібно?** Дослідник формулює твердження, створює необхідний показник і перевіряє його на підмножині даних.

`assign()` створює копію таблиці з додатковою колонкою, а `query()` дозволяє писати прості умови текстом. Це зручно для невеликих відтворюваних перевірок. Обережно: частка може бути не визначена, якщо знаменник дорівнює нулю.

**Приклад — виконайте комірку та розгляньте результат:**

In [ ]:
# Приклад: скільки записів з cnt > 0 у певному часовому проміжку.
active_evenings = (
    df.assign(is_evening=df["hr"].between(18, 21))
      .query("is_evening and cnt > 0")
)
print("Вечірніх рядків з ненульовими орендами:", len(active_evenings))

**✍️ Завдання 23 — тепер ви:**

1. Знайдіть усі рядки, де **`temp_c > 25`** та **`workingday == 0`**.  
2. Порахуйте середнє `cnt` у цій групі.  
3. Порівняйте із **неробочими годинами, де `temp_c <= 25`**.  
4. Чи достатньо такого порівняння, щоб зробити висновок про температуру?

> **Підказка:** Показуйте не лише середнє, а й `len(...)`, щоб бачити обсяг порівнюваних груп.

In [83]:
hot_days = df[
    (df["temp_c"] > 25) &
    (df["workingday"] == 0)
]

print(len(hot_days))
print(hot_days["cnt"].mean())

cool_days = df[
    (df["temp_c"] <= 25) &
    (df["workingday"] == 0)
]

print(len(cool_days))
print(cool_days["cnt"].mean())

801
343.3395755305868
4713
153.88372586462975


**Напишіть словами:** Назвіть хоча б два інші фактори, які могли впливати на побачену різницю.

_Ваша відповідь: ..._сезон року та погодні умови, сильний вітер

---
# Частина 6. Мініпроєкт: поясніть попит

Тепер спробуйте мислити як аналітик. Для кожної гіпотези потрібні: **що саме перевіряємо → як обчислюємо → які числа отримали → що вони означають → які є обмеження**.

## 24. Ваші три гіпотези (обов’язково)

Обирайте **три різні** твердження. Можете скористатись варіантами нижче **або вигадати свої**:

- **Пікові години** зареєстрованих користувачів відрізняються в робочі й неробочі дні.
- **Зв’язок температури й попиту** відрізняється за сезонами.
- У різні **погодні категорії** різниця між випадковими та зареєстрованими користувачами неоднакова.
- З 2011 до 2012 року середній погодинний попит виріс **не однаково в усі місяці**.
- У найхолодніші години спостерігається нижчий попит, але різниця залежить від часу доби.

**Для кожної гіпотези:** створіть окрему таблицю Pandas, запишіть числовий результат, сформулюйте висновок. Якщо гіпотеза не підтвердилася — це теж правильний результат дослідження.

In [88]:
# ГІПОТЕЗА 1
# Формулювання: ...Пікові години зареєстрованих користувачів відрізняються в робочі й неробочі дні
# Код перевірки: ...

peak_hours = []

for day_type in [0, 1]:
    hourly = (
        df[df["workingday"] == day_type]
        .groupby("hr")["registered"]
        .mean()
    )

    peak_hours.append({
        "workingday": day_type,
        "peak_hour": hourly.idxmax(),
        "avg_registered": hourly.max()
    })

peak_table = pd.DataFrame(peak_hours)

peak_table

# ГІПОТЕЗА 2
# Формулювання: ...Зв'язок температури й попиту відрізняється за сезонами
# Код перевірки: ...
#
# season_corr = []
#
# for season in sorted(df["season"].unique()):
#     data = df[df["season"] == season]
#
#     season_corr.append({
#         "season": season,
#         "observations": len(data),
#         "temperature_corr": data["temp_c"].corr(data["cnt"])
#     })
#
# season_table = pd.DataFrame(season_corr)
#
# season_table.round(3)

# ГІПОТЕЗА 3
# Формулювання: ...З 2011 до 2012 року середній погодинний попит виріс не однаково в усі місяці
# Код перевірки: ...
# monthly_demand = df.pivot_table(
#     index="mnth",
#     columns="yr",
#     values="cnt",
#     aggfunc="mean"
# )
#
# monthly_demand["difference"] = (
#     monthly_demand[1] - monthly_demand[0]
# )
#
# monthly_demand = monthly_demand.round(1)
#
# monthly_demand

,workingday,peak_hour,avg_registered
0,0,12,240.012987
1,1,17,468.394790


| № | Ваша гіпотеза | Які групи / метрики порівнювали | Результат із числами | Підтвердилася? | Що може спотворювати висновок?   |
|---|---|---|---|----------------|----------------------------------|
| 1 |	Пікові години зареєстрованих користувачів відрізняються в робочі й неробочі дні. | Середню кількість оренд зареєстрованими користувачами за годинами для workingday = 0 та workingday = 1.| Неробочі дні — 12:00 (240,01); робочі — 17:00 (468,39). | Так            | 	Свята, погода                              |
| 2 | 	Зв'язок температури й попиту відрізняється за сезонами. | Кореляцію між temp_c та cnt окремо для кожного сезону. | Весна — 0,446; літо — 0,408; осінь — 0,345; зима — 0,342. | Так              | Вологість, опади                 |
| 3 |З 2011 до 2012 року середній погодинний попит виріс не однаково в усі місяці. | Середню кількість оренд (cnt) за кожен місяць у 2011 та 2012 роках. | Найбільше зростання у березні — 134,2; найменше у грудні — 48,9. | Так            | Сезонність, погодні умови, свята |

## 25. Підсумковий звіт (обов’язково)

Напишіть 8–12 речень **своїми словами**, користуючись числами зі своїх розрахунків.

**1. Що ви дізналися про попит?** (час, день, сезон, погода)  
_Ваша відповідь: ..._Що попит залежить від місяця, сезону, для неділі та погоди. Наприклад, погода негативно може вплинути на попит, якщо вона негативна. Також, в період свят людей теж менше. Для попиту важлива хороша погода, робочий день та теплий сезон.

**2. Як поводяться `casual` та `registered`?**  
_Ваша відповідь: ..._Зареєстровані користувачі та випадкові користувачі мають різні особливості використання велосипедів. Зареєстровані користувачі можуть частіше використовувати велосипеди для поїздок, наприклад на роботу або навчання. Тому для них важливими можуть бути ранкові та вечірні години. Випадкові - для прогулянок.

**3. Які три практичні рішення ви запропонували б компанії?** (приклад напряму: коли збільшувати доступність велосипедів; не пишіть просто «покращити сервіс»)  
_Рекомендація 1: ..._ збільшувати доступність велосипедів приблизно о 8:00, 17:00 та 18:00, особливо в робочі дні, щоб задовольнити підвищений попит.
_Рекомендація 2: ..._  у теплі сезони та за сприятливої погоди забезпечувати більшу кількість доступних велосипедів, особливо у вихідні дні.
_Рекомендація 3: ..._окремо аналізувати попит зареєстрованих і випадкових користувачів та відповідно планувати розміщення велосипедів у популярних місцях у різний час доби.

**4. Які обмеження цього дослідження?** Згадайте принаймні **дві**: історичні дані, пропущені години, розмір груп, змішані фактори, кореляція ≠ причинність, відсутність інформації про індивідуальні поїздки.  
_Ваша відповідь: ..._По-перше, використовувалися історичні дані за 2011–2012 роки, тому вони можуть не відображати сучасність. По-друге, деякі дні містять менше ніж 24 погодинні записи, що може впливати на порівняння денної кількості оренд.Також на попит одночасно впливають температура, сезон, вітер, опади та інші фактори.

## Критерії оцінювання — 100 балів

| Розділ | Бали |
|---|---:|
| Частина 1: базові операції та пояснення (1–5) | 20 |
| Частина 2: фільтрація, якість, перетворення (6–11) | 25 |
| Частина 3: групування та порівняння (12–17) | 25 |
| Частина 4: кореляції, викиди, дати (18–21) | 15 |
| Три гіпотези з числовими доказами (24) | 10 |
| Підсумковий звіт і практичні висновки (25) | 5 |
| **Разом** | **100** |

**Бонус:** за кожне якісно виконане додаткове завдання 22 або 23 — **+5 балів** (максимум +10).

**Оцінюється не «красива відповідь», а логіка аналізу.** Навіть якщо результат виявився неочікуваним, добре обґрунтований висновок — це плюс.

## Шпаргалка: найчастіші операції Pandas

| Хочу зробити | Шаблон |
|---|---|
| Подивитись початок | `df.head(5)` |
| Дізнатись розмір | `df.shape` |
| Побачити типи | `df.info()` |
| Знайти статистики | `df["cnt"].describe()` |
| Порахувати категорії | `df["season"].value_counts()` |
| Вибрати кілька колонок | `df[["hr", "cnt"]]` |
| Відібрати рядки | `df[df["hr"] == 8]` |
| Поєднати умови | `df[(умова1) & (умова2)]` |
| Сортувати | `df.sort_values("cnt", ascending=False)` |
| Знайти пропуски | `df.isna().sum()` |
| Створити колонку | `df["new"] = ...` |
| Порівняти групи | `df.groupby("hr")["cnt"].mean()` |
| Декілька статистик | `df.groupby("hr").agg(...)` |
| Матриця середніх | `df.pivot_table(...)` |
| Частоти перетинів | `pd.crosstab(...)` |
| Кореляція | `df[["temp", "cnt"]].corr()` |
| Квартиль | `df["cnt"].quantile(0.75)` |
| Ковзне середнє | `daily["total_rentals"].rolling(7).mean()` |

### Перед здачею

- [ ] `hour.csv` покладено поруч з ноутбуком (або завантажено вручну в Colab).
- [ ] Усі обов’язкові завдання 1–21 виконані **кодом**, а не лише текстом.
- [ ] Після кожного завдання є коротке пояснення отриманих результатів.
- [ ] Є **три гіпотези**, їх перевірки та висновки.
- [ ] Є **три конкретні рекомендації** компанії.
- [ ] Ноутбук виконується від початку до кінця без помилок, якщо `hour.csv` лежить у папці проєкту.

**Не потрібно:** писати нейромережі, навчати моделі, встановлювати бібліотеки для графіків, шукати готові відповіді в інтернеті.